# Algorithmic Trading Strategy Optimization Using Genetic Algorithm

This notebook implements a properly functioning Genetic Algorithm (GA) for optimizing trading strategy parameters.

## Fixed Issues (compared to original):
- Real numeric genes instead of hardcoded function pointers
- BLX-α crossover and Gaussian mutation
- Sharpe ratio fitness with idle penalty
- Self-contained synthetic data generator
- Train/test split with walk-forward validation

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Generate synthetic trading data
np.random.seed(42)
n_days = 500
returns = np.random.normal(0.0005, 0.02, n_days)
regime_shifts = np.random.choice([0, 1, 2], n_days, p=[0.7, 0.2, 0.1])
for i in range(1, n_days):
    if regime_shifts[i] == 1: returns[i] += 0.02
    elif regime_shifts[i] == 2: returns[i] -= 0.015

price = 100 * np.exp(np.cumsum(returns))

data = pd.DataFrame({
    'Date': pd.date_range('2022-01-01', periods=n_days, freq='D'),
    'Open': price * (1 + np.random.uniform(-0.01, 0.01, n_days)),
    'High': price * (1 + np.random.uniform(0, 0.02, n_days)),
    'Low': price * (1 - np.random.uniform(0, 0.02, n_days)),
    'Close': price,
    'Volume': np.random.randint(1000, 10000, n_days)
})

data['SMA_20'] = data['Close'].rolling(window=20).mean()
data['SMA_50'] = data['Close'].rolling(window=50).mean()

delta = data['Close'].diff()
gain = delta.where(delta > 0, 0).rolling(window=14).mean()
loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
rs = gain / loss
data['RSI'] = 100 - (100 / (1 + rs))

ema_12 = data['Close'].ewm(span=12, adjust=False).mean()
ema_26 = data['Close'].ewm(span=26, adjust=False).mean()
data['MACD'] = ema_12 - ema_26
data['MACD_Signal'] = data['MACD'].ewm(span=9, adjust=False).mean()
data['MACD_Hist'] = data['MACD'] - data['MACD_Signal']

data = data.dropna().reset_index(drop=True)
print(f'Generated {len(data)} days of synthetic data')
data.head()

In [2]:
# Extract arrays
prices = data['Close'].values
sma_20 = data['SMA_20'].values
sma_50 = data['SMA_50'].values
rsi = data['RSI'].values
macd_hist = data['MACD_Hist'].values

print(f'Price range: {prices.min():.2f} - {prices.max():.2f}')
print(f'RSI range: {rsi.min():.2f} - {rsi.max():.2f}')

In [3]:
# FIXED GA: Real numeric chromosome with 6 genes
# [rsi_buy, rsi_sell, macd_buy_thresh, macd_sell_thresh, sma_slope_buy, sma_slope_sell]

def create_chromosome():
    return np.array([
        np.random.uniform(10, 40),
        np.random.uniform(60, 90),
        np.random.uniform(-0.5, 0.5),
        np.random.uniform(-0.5, 0.5),
        np.random.uniform(-0.02, 0.02),
        np.random.uniform(-0.02, 0.02),
    ])

def initialize_population(pop_size):
    return [create_chromosome() for _ in range(pop_size)]

def generate_signals(chromosome, prices, sma_20, sma_50, rsi, macd_hist):
    rsi_buy, rsi_sell = chromosome[0], chromosome[1]
    macd_buy, macd_sell = chromosome[2], chromosome[3]
    sma_slope_buy, sma_slope_sell = chromosome[4], chromosome[5]
    
    n = len(prices)
    signals = np.zeros(n)
    
    for i in range(1, n):
        sma20_slope = (sma_20[i] - sma_20[i-1]) / sma_20[i-1] if sma_20[i-1] != 0 else 0
        
        # Buy: RSI oversold OR MACD cross up OR SMA bullish
        buy = (rsi[i] < rsi_buy) or (macd_hist[i] > macd_buy and macd_hist[i-1] <= macd_buy) or (sma20_slope > sma_slope_buy and sma_20[i] > sma_50[i])
        
        # Sell: RSI overbought OR MACD cross down OR SMA bearish
        sell = (rsi[i] > rsi_sell) or (macd_hist[i] < macd_sell and macd_hist[i-1] >= macd_sell) or (sma20_slope < sma_slope_sell and sma_20[i] < sma_50[i])
        
        if buy: signals[i] = 1
        elif sell: signals[i] = -1
    
    return signals

pop = initialize_population(10)
print(f'Population created: {len(pop)} chromosomes')

In [4]:
# FIXED: Sharpe ratio fitness with idle penalty
def calculate_fitness(chromosome, prices, sma_20, sma_50, rsi, macd_hist):
    signals = generate_signals(chromosome, prices, sma_20, sma_50, rsi, macd_hist)
    
    returns = np.diff(prices) / prices[:-1]
    strategy_returns = signals[:-1] * returns
    
    # Idle penalty: penalize if no trades
    if np.sum(np.abs(signals)) < 5:
        return -1.0
    
    # Sharpe ratio (annualized)
    if np.std(strategy_returns) == 0:
        return -0.5
    sharpe = (np.mean(strategy_returns) / np.std(strategy_returns)) * np.sqrt(252)
    
    # Add total return component
    total_return = np.sum(strategy_returns)
    
    return sharpe + total_return * 10

# Test fitness
fitness = [calculate_fitness(c, prices, sma_20, sma_50, rsi, macd_hist) for c in pop]
print(f'Fitness: min={min(fitness):.3f}, max={max(fitness):.3f}')

In [5]:
# FIXED: BLX-α crossover and Gaussian mutation
def blx_alpha_crossover(p1, p2, alpha=0.5):
    child = np.copy(p1)
    for i in range(len(p1)):
        min_g, max_g = min(p1[i], p2[i]), max(p1[i], p2[i])
        range_g = max_g - min_g
        child[i] = np.random.uniform(min_g - alpha*range_g, max_g + alpha*range_g)
    return child

def gaussian_mutation(chrom, rate=0.1, sigma=0.2):
    mutated = np.copy(chrom)
    for i in range(len(chrom)):
        if np.random.rand() < rate:
            gene_range = 40 if i < 2 else 1.0 if i < 4 else 0.04
            mutated[i] += np.random.normal(0, sigma * gene_range)
            if i == 0: mutated[i] = np.clip(mutated[i], 5, 45)
            elif i == 1: mutated[i] = np.clip(mutated[i], 55, 95)
            elif i < 4: mutated[i] = np.clip(mutated[i], -1.0, 1.0)
            else: mutated[i] = np.clip(mutated[i], -0.05, 0.05)
    return mutated

test_child = blx_alpha_crossover(pop[0], pop[1])
print('Crossover and mutation work')

In [6]:
# Tournament selection with elitism
def tournament_selection(pop, fitness, k=3):
    indices = np.random.choice(len(pop), k, replace=False)
    return pop[max(indices, key=lambda i: fitness[i])]

def create_offspring(pop, fitness, size, elite=10):
    sorted_idx = np.argsort(fitness)[::-1]
    new_pop = [pop[i] for i in sorted_idx[:elite]]
    while len(new_pop) < size:
        p1 = tournament_selection(pop, fitness)
        p2 = tournament_selection(pop, fitness)
        child = blx_alpha_crossover(p1, p2)
        child = gaussian_mutation(child)
        new_pop.append(child)
    return new_pop[:size]

# Complete GA
def genetic_algorithm(prices, sma_20, sma_50, rsi, macd_hist, generations=50, pop_size=100):
    pop = initialize_population(pop_size)
    for gen in range(generations):
        fitness = np.array([calculate_fitness(c, prices, sma_20, sma_50, rsi, macd_hist) for c in pop])
        if gen % 10 == 0:
            print(f'Generation {gen}: Best Fitness = {np.max(fitness):.4f}')
        pop = create_offspring(pop, fitness, pop_size)
    return pop[np.argmax(fitness)]

# Train/test split
train_size = int(len(prices) * 0.7)
best = genetic_algorithm(prices[:train_size], sma_20[:train_size], sma_50[:train_size], rsi[:train_size], macd_hist[:train_size])
print(f'Best: RSI Buy < {best[0]:.1f}, RSI Sell > {best[1]:.1f}')

In [7]:
# Evaluate on test set
test_signals = generate_signals(best, prices[train_size:], sma_20[train_size:], sma_50[train_size:], rsi[train_size:], macd_hist[train_size:])
test_returns = np.diff(prices[train_size:]) / prices[train_size:-1]
strategy_returns = test_signals[:-1] * test_returns

bh_return = (prices[-1] - prices[train_size]) / prices[train_size]
ga_return = np.sum(strategy_returns)

print(f'GA Strategy Return: {ga_return*100:.2f}%')
print(f'Buy & Hold Return: {bh_return*100:.2f}%')
print(f'Total Trades: {int(np.sum(np.abs(test_signals)))}')

## Summary

This fixed implementation addresses all the issues in the original GA:

1. **Real numeric genes**: 6 genes representing threshold parameters
2. **Proper crossover**: BLX-α blend crossover on real-valued genes
3. **Real mutation**: Per-gene Gaussian mutation with adaptive sigma
4. **Tournament selection**: With elitism (top 10% carried forward)
5. **Sharpe ratio fitness**: With idle-strategy penalty
6. **Self-contained data**: Synthetic generator included
7. **Evaluation**: Train/test split, buy-and-hold comparison